In [ ]:
import cvxpy as cp
import numpy as np

#Function for Partial Trace

def partial_trace_one_qubit(rho, trace_qubit, num_qubits):
    """
    Compute the partial trace over one qubit in a 4-qubit system (16x16 density matrix),
    symbolically and CVXPY-compatible.

    Args:
        rho: cp.Variable or cp.Constant of shape (16, 16)
        trace_qubit: int (0 to 3), index of qubit to trace out (0 = most significant)

    Returns:
        cp.Expression of shape (8, 8): reduced density matrix after partial trace
    """

    full_dim = 2 ** num_qubits      # 16
    reduced_dim = 2 ** (num_qubits - 1)  # 8

    # Initialize symbolic 8x8 matrix
    reduced_rho = [[0 for _ in range(reduced_dim)] for _ in range(reduced_dim)]

    for i in range(full_dim):
        for j in range(full_dim):
            # Convert i, j to binary lists (length 4)
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]

            # Only keep entries where traced-out qubit matches in bra & ket
            if bi[trace_qubit] == bj[trace_qubit]:
                # Remove the traced qubit
                bi_reduced = bi[:trace_qubit] + bi[trace_qubit+1:]
                bj_reduced = bj[:trace_qubit] + bj[trace_qubit+1:]

                # Convert reduced binary back to integer index
                row = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bi_reduced)])
                col = sum([bit << (num_qubits - 2 - k) for k, bit in enumerate(bj_reduced)])

                # Accumulate entry
                reduced_rho[row][col] += rho[i, j]

    # Convert to cvxpy Expression matrix
    reduced_rho_expr = cp.vstack([cp.hstack(row) for row in reduced_rho])
    return reduced_rho_expr



#Function for Partial Tranpose

def partial_transpose_qubits(rho, transpose_qubits, num_qubits):
    """
    Compute the partial transpose of a 4-qubit (16x16) density matrix with respect to a
    subset of qubits, symbolically and CVXPY-compatible.

    Args:
        rho: cp.Variable or cp.Constant of shape (16, 16)
        transpose_qubits: list of ints (0-based), e.g., [1, 2] for second and third qubits

    Returns:
        A cp.Expression (16x16) representing the partial transpose of rho
    """
    
    dim = 2 ** num_qubits

    # Initialize result matrix
    rho_pt = [[0 for _ in range(dim)] for _ in range(dim)]

    for i in range(dim):
        for j in range(dim):
            # Convert i and j to binary basis labels (length 4)
            bi = [(i >> k) & 1 for k in reversed(range(num_qubits))]
            bj = [(j >> k) & 1 for k in reversed(range(num_qubits))]

            # Apply transpose to selected qubits: swap bi[k] and bj[k] if k in transpose_qubits
            bi_new = bi.copy()
            bj_new = bj.copy()
            for q in transpose_qubits:
                bi_new[q], bj_new[q] = bj[q], bi[q]

            # Convert new binary indices to flattened integer
            row_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bi_new)])
            col_new = sum([bit << (num_qubits - 1 - k) for k, bit in enumerate(bj_new)])

            # Set rho_pt[row_new, col_new] = rho[i, j]
            rho_pt[row_new][col_new] += rho[i, j]

    # Convert to CVXPY Expression matrix
    return cp.vstack([cp.hstack(row) for row in rho_pt])

# Function for Random Parameters
def generate_sorted_unit_sum_set(n=4):
    while True:
        # Generate n random numbers in (0,1)
        values = np.random.rand(n)
        # Normalize them to sum to 1
        values /= values.sum()
        # Sort in descending order
        sorted_values = np.sort(values)[::-1]
        # Ensure all values are > 0 and < 1
        if all(0 < x < 1 for x in sorted_values):
            return sorted_values
        
# Generate random parameters        

p = generate_sorted_unit_sum_set()
q = generate_sorted_unit_sum_set()

p1, p2, p3, p4 = p
q1, q2, q3, q4 = q
K = 8  # Number of M_i operators (adjust as needed)

#Analytical bound on fidelity
F_anal=max(1/2,p1*q1+p2*q2+p3*q3+p4*q4,q1*p2+q2*p1+q3*p4+q4*p3,q1*p3+q2*p4+q3*p1+q4*p2,q1*p4+q2*p3+q3*p2+q4*p1)
# Define rho_{12} (4x4 matrix)
rho_12 = np.array([
    [(p1 + p2)/2, 0, 0, (p1 - p2)/2],
    [0, (p3 + p4)/2, (p3 - p4)/2, 0],
    [0, (p3 - p4)/2, (p3 + p4)/2, 0],
    [(p1 - p2)/2, 0, 0, (p1 + p2)/2]
])

# Define sigma_{34} (4x4 matrix)
sigma_34 = np.array([
     [(q1 + q2)/2, 0, 0, (q1 - q2)/2],
    [0, (q3 + q4)/2, (q3 - q4)/2, 0],
    [0, (q3 - q4)/2, (q3 + q4)/2, 0],
    [(q1 - q2)/2, 0, 0, (q1 + q2)/2]
])

# Construct rho_{12} ⊗ sigma_{34} (16x16 matrix)
rho_sigma = np.kron(rho_12, sigma_34)

# Define M_i as 16x16 PSD variables
M = [cp.Variable((16, 16), hermitian=True) for _ in range(K)]

# Constraints
constraints = []

# Constraint 1: 2 sum_i (Tr_4(M_i) ⊗ I_2) = I_{16}
identity_16 = np.eye(16)
identity_2 = np.eye(2)  # I_2 for subsystem 2 (since Tr_4 acts on subsystem 4)

# Compute Tr_4(M_i) for each M_i (result is 8x8 matrix)
# Then take Kronecker product with I_2
sum_Tr4_Mi_tensor_I2 = 0
for M_i in M:
    Tr4_Mi = partial_trace_one_qubit(M_i, trace_qubit=3, num_qubits=4)  # Partial trace over subsystem 4 (last qubit)
    Tr4_Mi_tensor_I2 = cp.kron(Tr4_Mi, identity_2)
    sum_Tr4_Mi_tensor_I2 += Tr4_Mi_tensor_I2
p_expr = 2 * cp.real(cp.trace(sum_Tr4_Mi_tensor_I2 @ rho_sigma))

constraints.append(2 * cp.real(sum_Tr4_Mi_tensor_I2) <= identity_16)

# Constraint 2: 0 ≤ M_i ≤ I_{16}
for M_i in M:
    constraints.append(M_i >> 0)
    constraints.append(M_i << identity_16)

# Constraint 3: M_i^{T_{23}} ⪰ 0 (partial transpose on subsystems 2 & 3)
for M_i in M:
    M_i_T23=partial_transpose_qubits(M_i, transpose_qubits=[1,2], num_qubits=4)
    constraints.append(M_i_T23 >> 0) 

# Objective

objective_terms = sum(cp.real(cp.trace(M_i @ rho_sigma)) for M_i in M)
objective = cp.Maximize((1 - p_expr)/2 + objective_terms)

# Solve the SDP
problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.MOSEK, verbose=True)

# Print results
print("Optimal value:", problem.value)
print("p value:", p_expr.value)
print("Objective term:", objective_terms.value)


#Print deviation from analytical bound
print("Deviation from analytical bound:", problem.value - F_anal)

# Print parameters
print(f"Set P: p1={p1}, p2={p2}, p3={p3}, p4={p4}, Sum={p.sum()}")
print(f"Set Q: q1={q1}, q2={q2}, q3={q3}, q4={q4}, Sum={q.sum()}")

#Individual eigenvalues
print("p1*q1+p2*q2+p3*q3+p4*q4=",p1*q1+p2*q2+p3*q3+p4*q4)
print("q1*p2+q2*p1+q3*p4+q4*p3=",q1*p2+q2*p1+q3*p4+q4*p3)
print("q1*p3+q2*p4+q3*p1+q4*p2=",q1*p3+q2*p4+q3*p1+q4*p2)
print("q1*p4+q2*p3+q3*p2+q4*p1=",q1*p4+q2*p3+q3*p2+q4*p1)
#for i in range(K):
#    print(f"M_{i} (16x16):\n", M[i].value)  # May need rounding for readability